<a href="https://colab.research.google.com/github/Stubberson/project-collection/blob/main/file_sizes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Retrieving File Sizes for Map Overlays
This notebook retireves the file sizes for each map overlay that's been used in questionnaire. For zip files it retrieves the uncompressed folder size through the central directory. The URLs for overlays can be retrieved through Superset.

In [ ]:
import pandas as pd

df_overlay = pd.read_csv('/content/overlay_URLs.csv')

print("df_overlay head:")
display(df_overlay.head())

# CSVs to Dataframes
Read the file "URLs.csv" into a dataframe and add a new column with the file sizes of the URLs listed in the 'urls' column.

In [ ]:
import numpy as np
import ast

# Function to safely evaluate the string representation of a list or return an empty list
def safe_eval_list(x):
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except (ValueError, SyntaxError):
            return []
    elif isinstance(x, list):
        return x
    else:
        return []

# Apply safe_eval_list to each url column
df_overlay['urls'] = df_overlay['urls'].apply(safe_eval_list)

## Define and apply functions to get file sizes

Define and apply functions to retrieve the file size for each URL, handling potential errors.

* `get_file_size(url)` gets the size of a single URL
* `get_file_sizes_for_list(urls_list)` gets all the file sizes within the URL lists

I'm retireving the uncompressed size of a zip file with the `remotezip` library and looking into the "central directory" of the zip file. This technique relies on the web server hosting the file supporting HTTP Range Requests. This feature allows a client to request only a specific portion (a "range" of bytes) of a file. Most modern web servers and cloud storage services (like AWS S3, Google Cloud Storage, etc.) support this. Maptionnaire uses AWS Cloudfront as the cloud storage platform, so this is possible.

In [ ]:
# Use remotezip to access the cenral directory
!pip install remotezip

In [ ]:
import requests
from remotezip import RemoteZip

def get_file_size(url):
    """
    Retrieves the file size for a given URL.
    For .zip files, it attempts to get the uncompressed size from the .zip files central directory using remotezip.

    Args:
        url: The URL of the file.

    Returns:
        The file size in bytes as an integer, or None if the size cannot be retrieved.
    """
    try:
        if url.lower().endswith('.zip'):
            try:
                with RemoteZip(url) as rz:
                    total_uncompressed_size = 0
                    for file_info in rz.infolist():
                        total_uncompressed_size += file_info.file_size
                    return total_uncompressed_size
            except Exception as e:
                print(f"Error processing zip file {url}: {e}")
                return None # Return None if remotezip fails
        else:
            response = requests.head(url)
            if response.status_code == 200:
                content_length = response.headers.get('Content-Length')
                if content_length:
                    return int(content_length)
            return None
    except requests.exceptions.RequestException as e:
        print(f"Error fetching {url}: {e}")
        return None

## Apply the function
Apply the `get_file_size` function to each list of urls in the 'urls' column of the dataframe to get the file sizes for each questionnaire.
```
Process:
  1. define a function that takes a list of urls as input and iterates through each url in the list.
  2. for each url in the list, call the `get_file_size` function to get its size.
  3. store the retrieved file sizes in a list.
  4. return the list of file sizes.
  5. apply this new function to the 'urls' column of the `df` dataframe using the `.apply()` method.
  6. store the resulting list of file sizes for each row in a new temporary column in the `df` dataframe.
```


In [ ]:
def get_file_sizes_for_list(urls_list):
    """
    Retrieves file sizes for a list of URLs.

    Args:
        urls_list: A list of URLs.

    Returns:
        A list of file sizes (integers or None) for each URL in the input list.
    """
    sizes = []
    for url in urls_list:
        sizes.append(get_file_size(url))
    return sizes

df_overlay['file_sizes'] = df_overlay['urls'].apply(get_file_sizes_for_list)

In [ ]:
# Apply summation and display to check everything's ok
df_overlay['total_file_size'] = df_overlay['file_sizes'].apply(lambda x: sum([s for s in x if s is not None]))
display(df_overlay.head())

In [ ]:
df_overlay.to_csv('file_sizes.csv', index=False)